# ASR demo

DeepSpeech2 speech recognition: reproduce the reported result and transcribe your own dataset.

Run in a fresh Google Colab session with a GPU enabled, then execute the cells in order.

## 1. Install dependencies

The project uses uv for dependency management. In Colab, install the exported
`requirements.txt` into the current notebook environment with `%pip`.
Use Python 3.13, matching the project environment.


In [ ]:
from google.colab import userdata
GIT_TOKEN = userdata.get('GIT_TOKEN')

# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# GIT_TOKEN = user_secrets.get_secret("GIT_TOKEN")

repo_url = f"https://{GIT_TOKEN}@github.com/DogeSavior3/DLA-ASR-project.git"

!git clone {repo_url}

In [ ]:
# %git clone {"https://github.com/DogeSavior3/DLA-ASR-project.git"}

from pathlib import Path
import os
import shutil

for file in Path('DLA-ASR-project').iterdir():
    shutil.move(str(file), str(Path('DLA-ASR-project').parent / file.name))
os.rmdir(Path('DLA-ASR-project'))

!wget -c --progress=bar:force:noscroll \
  https://openslr.elda.org/resources/12/test-other.tar.gz \
  -O /content/test-other.tar.gz

!mkdir -p data/datasets/librispeech
!tar -xzf /content/test-other.tar.gz \
  -C data/datasets/librispeech \
  --strip-components=1 \
  LibriSpeech/test-other

%pip install -qr requirements.txt

## 2. Download the model

Download the checkpoint and extract its model weights for `inference.py`.
No training is required.


In [ ]:
import torch
from huggingface_hub import hf_hub_download, login
HUGGINGFACE_TOKEN = userdata.get("HUGGINGFACE_TOKEN")
login(HUGGINGFACE_TOKEN)

hub_model_id = "DogeSavior/DLA-ASR_deepspeech2"
checkpoint_path = hf_hub_download(
    repo_id=hub_model_id,
    filename="model_final.pth",
)
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

weights_path = "ds2_pretrained_weights.pth"
torch.save(checkpoint["state_dict"], weights_path)
del checkpoint


## 3. Reproduce the reported result

Run inference on the complete LibriSpeech test-other split.
The script prints CER/WER for each decoding method and saves predictions
under `data/saved/reproduct/test_other/`.

In [ ]:
!python -u inference.py --config-name=inference_reproduct \
  inferencer.from_pretrained="{weights_path}"

## 4. Evaluate your own dataset

Download and extract your dataset first, then set `DATASET_DIR` to its directory.
It should contain `audio/` and, optionally, `transcriptions/` with matching IDs:
`audio/sample.wav` and `transcriptions/sample.txt`.

The script scores only recordings with reference transcriptions, and saves
argmax and beam predictions for every recording.


In [ ]:
#DATASET_DIR = ""  # path to the extracted directory containing audio/.
# or get it from google drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_DIR = 'drive/MyDrive/Colab_Notebooks/DLA/LibriSpeech'

In [ ]:
dataset_dir = Path(DATASET_DIR)

!python inference.py --config-name=inference-custom \
  inferencer.from_pretrained="{weights_path}" \
  dataset_dir="{dataset_dir}" \
  inferencer.beam_size=3
